### Exploring sythetic prompt generation as prompt protection

In this notebook, you'll work on finding ways to create synthetic prompts that mirror original prompts. You'll use a few different methods to do so and see both some potential use cases but also where this strategy might fail. 

Before running this notebook, you'll need to install a [llamafile](https://github.com/Mozilla-Ocho/llamafile) and start it in the background by:

1. Making the file executable
2. Running this executable

It should open a browser. We'll use it from here though!

For the later part of the notebook, you'll either use [Hugging Face models](https://huggingface.co/) (if you have an account and also a computer that can run them) or [ollama](https://ollama.com/). For ollama, you can also [install the software](https://ollama.com/) if you'd like to do more local-first AI, but it should also Just Work with the pip library.

In [23]:
!pip install dotenv ollama

  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.11.7-py3-none-any.whl (444 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.8 MB/s eta 0:00:0031m12.5 MB/s eta 0:00:01
Using cached typing_inspection-0.4.1-py3-none-any.whl (14 kB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.18.4
    Uninstalling pydantic_core-2.18.4:
      Successfully uninstalled pydantic_core-2.18.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.7.4
    Uninstalling pydantic-2.7.4:
      Successfully uninstalled pydantic-2.7.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
great-expectations 0.18.16 requires numpy<2.0.0,>=1.22.4; python_version >= "3.10", but you have numpy 2.2.6 which is incompa

In [24]:
import llamafile_client as llamafile
import ollama
import torch
from time import time
from datetime import timedelta

### Initial prompt engineering

Our goal is to see just how far we can get from prompt engineering alone. Can prompt engineering help us sanitize and pseudonymize prompts? Let's try it out with our llamafile and writing some clever prompts. You can reuse any of the prompts you used in the Presidio notebook or take a look at the "Master Class - Generating Sensitive Prompts" notebook or data/sensitive_prompts.json 

First, let's start with something that mirrors the Presidio task.


In [15]:
initial_prompt = """
Take the text below.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Here is the text to fix:

{text}
"""

In [16]:
sensitive_text = "His name is Mr. Samuel Jones and his phone number is +49 110 0342121345"

In [17]:
initial_prompt.format(text=sensitive_text)

'\nTake the text below.\nLocate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. \nReplace these sensitive tokens with a REDACTED token. \n\nHere is the text to fix:\n\nHis name is Mr. Samuel Jones and his phone number is +49 110 0342121345\n'

In [18]:
first_output = llamafile.completion(initial_prompt.format(text=sensitive_text))

print(first_output)


Replace Mr. Samuel Jones with a [REDACTED] name.
Replace +49 110 0342121345 with a [REDACTED] phone number.


The text is now fixed!</s>


Thanks, I think... lol. 😂

Give it a try with a few other texts, does it at least find the PII tokens? 

# Leveraging Few-shot / In-Context Learning

Few-shot and in-context learning is a way to on-the-fly generate different responses or to recontextualize the in-memory information to better perform a task. Often this is used to complete simple language-tasks or other tasks by showing examples, but it can also be used to write better prompts.

To invoke in-context, usually multiple examples are shown. For example:

```
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
```

In [19]:
in_context_prompt = """
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
"""

In [20]:
more_text = """
Yo, can you please call me back -- you have my number under JJ already. It's urgent because your dad is in the hospital and they are asking for his identity details. I htink his SS number starts with 444-22- but I forget if the last numbers are 5543 or 5544. Call me back !"""

In [22]:
second_output = llamafile.completion(in_context_prompt.format(text=more_text))

print(second_output)


You have my number under [REDACTED]. It's urgent because your dad is in the hospital and they are asking for his identity details. I htink his SS number starts with 444-22- but I forget if the last numbers are [REDACTED] or [REDACTED]. Call me back !</s>


Getting better... now you try with a few additional in-context examples AND some more tricky text!

## Crescendo attack as inspiration

Microsoft released [a paper and adversarial approach](https://crescendo-the-multiturn-jailbreak.github.io/) for jailbreaking LLMs called Crescendo. In this approach, the attacker tries to generate increasingly adversarial prompts pointing the entire chat history/conversation slowly into what would before be banned content (i.e. illegal, explicit, harrassment/hate speech.)

To automate this, the attack utilizes more than one LLM at once to steer the conversation towards some goal. 

The idea is:

1. Setup one model as the model you want to attack. You can do that via HuggingFace/llamafile or via a model running on the LAN. 
2. Setup one model as the model that will help you attack the other model to produce increasingly more feminist responses. You can do that via HuggingFace/llamafile or via a model running on the LAN.
3. (optional): if you need a model to review potential guardrails, you might want to try getting [LlamaGuard](https://github.com/meta-llama/PurpleLlama) running. Note that this will require 10s of gigs of memory. (I know :()
4. Start manually and guide one conversation to the next. One tip from the research is to have the helper model summarize the other model's response and then nudge toward the goal.
5. If it works, you can try to see if there is an automatable attack template. Check out the research for the pseudocode/example.


We are going to use it to try to better sanitize our prompts. Let's go!

In [26]:
ollama_client = ollama.Client()
model_name = 'gemma3:4b'

In [29]:
ollama_response = ollama_client.chat(
    model=model_name,
    messages=[
        {'role': 'user', 
         'content': 'is this thing on?'}]
)
print(ollama_response)

model='gemma3:4b' created_at='2025-08-07T06:53:36.596112Z' done=True done_reason='stop' total_duration=5335468334 load_duration=2205273167 prompt_eval_count=14 prompt_eval_duration=1257638958 eval_count=50 eval_duration=1871989000 message=Message(role='assistant', content='Yes! Absolutely! This thing is definitely on! 😄\n\nHow can I help you today? 😊 \n\nDo you want to:\n\n*   Chat about something?\n*   Ask me a question?\n*   Play a game?', thinking=None, images=None, tool_name=None, tool_calls=None)


In [32]:
crescendo_init_prompt = """
Rewrite texts to remove potentially sensitive tokens.
Locate names, addresses, URLs, phone numbers, email addresses, identity numbers, dates, credit card numbers, bank account numbers, cryptographic keys or usernames. 
Locate partial identifiers, like partial ID numbers or things like family relationships.
Replace these sensitive tokens with a REDACTED token. 

Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
"""


In [33]:
cres_first_output = llamafile.completion(crescendo_init_prompt.format(text=sensitive_text))

print(cres_first_output)


Redacted:

Mr. Samuel Jones's phone number is [REDACTED].
Janet arrived at the emergency room [REDACTED] and was immediately taken into the ICU. If you want to reach the nurses line directly call [REDACTED].
Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].

Note that redaction is a process of removing sensitive information from texts. It is done to protect individuals' privacy and security. There are different ways to redact information, but the most common method is to replace sensitive tokens with a placeholder or a REDACTED token. The examples above show how to redact names, phone numbers, and addresses.</s>


In [35]:
cres_discriminator_prompt = """

Your job is to find errors in sensitive token redaction. Sensitive tokens include things like:

- names
- phone numbers
- identity numbers
- partial identity numbers
- credit card numbers 
- partial names
- family relationships
- dates
- credit card numbers
- bank account numbers
- cryptographic keys
- usernames

This data should be replaced by a [REDACTED] token.

Review the following text: 

{text}

Summarize the redaction errors and give a better prompt to guide how to redact sensitive information based on the errors seen. 
"""


In [37]:
messages = [
    {'role': 'user',
     'content': crescendo_init_prompt
    }, 
    {
     'role': 'agent',
     'content': cres_first_output,
    },
    {'role': 'user',
     'content': cres_discriminator_prompt.format(text=cres_first_output),
    }
]
     

In [43]:
cres_disc_response = ollama_client.chat(
    model=model_name,
    messages=messages,
)
print(cres_disc_response.message.content)

Okay, here's an analysis of the redaction errors and a revised prompt to guide the process:

**Redaction Errors Identified:**

*   **Inconsistent Redaction:** The initial text contained multiple instances of “[REDACTED]” without clear justification. It wasn’t immediately obvious *what* was being redacted in each case.
*   **Missing Contextualization:** The redaction lacked context. It simply replaced strings with “[REDACTED]” without explaining *why* those strings were sensitive.
*   **Over-Redaction:**  “Its tied to my same last name [REDACTED]” is a reasonable redaction because family relationships can be considered sensitive. However, the prompt doesn't provide guidance on when to redact family relationships. 


**Revised Prompt:**

“Your job is to meticulously identify and redact sensitive tokens from text. Sensitive tokens include, but are not limited to:

*   **Names:** Full names, partial names.
*   **Phone Numbers:** All phone numbers, including area codes.
*   **Identity Numbe

In [50]:
disc_prompt = cres_disc_response.message.content.split('---')[0] + """
Follow these examples:

His name is Mr. Samuel Jones and his phone number is +49 110 0342121345 // His name is [REDACTED] and his phone number is [REDACTED]
Sure, my drivers license is B4382947-8. Its tied to my same last name Yarok. // Sure, my drivers license is [REDACTED]. Its tied to my same last name [REDACTED].
Janet arrived at the emergency room yesterday evening at 11:24pm and was immediately taken into the ICU. Her vitals are stable and her daughter Jessica stayed overnight to keep an eye on her. If you want to reach the nurses line directly call 777-1138. // [REDACTED] arrived at the emergency room [REDACTED] and was immediately taken into the ICU. Her vitals are stable and [REDACTED] stayed overnight to keep an eye on her. If you want to reach the nurses line directly call [REDACTED]. 
{text} //
"""

In [51]:
disc_prompt

"Okay, here's an analysis of the redaction errors and a revised prompt to guide the process:\n\n**Redaction Errors Identified:**\n\n*   **Inconsistent Redaction:** The initial text contained multiple instances of “[REDACTED]” without clear justification. It wasn’t immediately obvious *what* was being redacted in each case.\n*   **Missing Contextualization:** The redaction lacked context. It simply replaced strings with “[REDACTED]” without explaining *why* those strings were sensitive.\n*   **Over-Redaction:**  “Its tied to my same last name [REDACTED]” is a reasonable redaction because family relationships can be considered sensitive. However, the prompt doesn't provide guidance on when to redact family relationships. \n\n\n**Revised Prompt:**\n\n“Your job is to meticulously identify and redact sensitive tokens from text. Sensitive tokens include, but are not limited to:\n\n*   **Names:** Full names, partial names.\n*   **Phone Numbers:** All phone numbers, including area codes.\n*   

In [52]:
cres_third_output = llamafile.completion(disc_prompt.format(text=sensitive_text))

In [53]:
cres_third_output

'\n\n5. **Avoid Reproducing Sensitive Data:** Avoid reproducing any sensitive data, e.g. social security numbers, names of family members, dates of birth etc. While redacting data, avoid introducing any other information that may lead to the identification of sensitive data. Do not duplicate sensitive data by accident.\n\n**Example:** Do not redact by saying "Mr. Samuel Jones\'s phone number is [REDACTED] but Mr. Samuel Jones is still Mr. Samuel Jones." Instead, redact Mr. Samuel Jones\'s phone number entirely. If you have a choice between redacting sensitive data and introducing data that may lead to the identification of sensitive data, choose to redact sensitive data.\n\n6. **Use the Principle of Least Privilege:** Only redact data that is necessary. Do not redact more data than required. If you have access to sensitive data, make sure to redact it completely and don\'t share it with unauthorized parties.\n\n**Example:** If you have access to sensitive data such as social security n

Ruh roh... welllppp, now it's your turn, can you find better success? :)

### Hugging Face Addendum! :) 

Note: the code in the next cell is only if you have a GPU or a M1/M2 silicon.. if not, please just use more llamafiles or a LAN model. Alternatively, you can set up and use an API for a popular LLM.

- If you have one or more GPUs, please change your device below to map to your GPU, sometimes just "cuda" works.
- If you have a Macbook M1 or M2, please make sure you have [pytorch for MPS installed](https://developer.apple.com/metal/pytorch/).

To use the meta-llama repository you must also:

1. Create a HuggingFace account.
2. Install hugging face and cli: `pip install -U "huggingface_hub[cli]"`
3. Request access to the meta-llama3 models, which you can do [on the model page](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). Once they approve you will get an email.
5. Log into hugging face CLI using your user token (create one in User->Settings). Set the token to Read to just download models. Command is `huggingface-cli login`.
6. Then proceed with code below!

Continue or switch models and way that you are steering. Work with a partner to explore what paths you find interesting!

In [11]:
# Only run this after reading above note! :)
from transformers import pipeline


pipe = pipeline("text-generation", 
                model="meta-llama/Llama-3.2-1B-Instruct", # you can also choose other huggingface models here!
                torch_dtype=torch.bfloat16, 
                device="mps", 
                max_new_tokens=500) # here is where to change your device if you use something other than mac silicon

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]